VERSIÓ 1

In [1]:
import requests, pandas as pd, re, time
from bs4 import BeautifulSoup
from datetime import datetime

# -------- CONFIG --------
USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
CIK = "0000320193"  # Apple; keep leading zeros
FORMS = {"10-K", "10-Q"}
START = "2001-09-01"
END   = "2025-08-14"
PAUSE = 0.2

LABELS = {
    "revenue": [r"^revenue", r"net sales", r"sales,? net"],
    "net_income": [r"net income", r"net (loss|income)", r"profit attributable.*?shareholders"],
    "assets": [r"total assets"],
    "liabilities": [r"total liabilities"],
    "equity": [r"(total )?(share|stock)holders[’']? equity", r"total (members’|partners’) equity", r"total equity"]
}

# ----- UTIL -----
def parse_number(s: str):
    if s is None: return None
    neg = bool(re.search(r"\(\s*[\d,\.]+\s*\)", s))
    t = re.sub(r"[^\d\.\-]", "", s)
    if t in {"", "-", "."}: return None
    try:
        val = float(t)
        return -val if neg and val > 0 else val
    except:
        return None

def in_range(date_str):
    d = datetime.strptime(date_str, "%Y-%m-%d").date()
    return datetime.strptime(START, "%Y-%m-%d").date() <= d <= datetime.strptime(END, "%Y-%m-%d").date()

def pick_primary_doc_url(cik_int, accession_no, primary_doc):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_no.replace('-', '')}/{primary_doc}"

def find_best_number_in_row(cells):
    nums = [parse_number(c) for c in cells if re.search(r"\d", c or "")]
    nums = [n for n in nums if n is not None]
    return nums[-1] if nums else None

def label_matches(text, patterns):
    t = (text or "").lower()
    for p in patterns:
        if re.search(p, t, re.I):
            return True
    return False

def extract_metrics_from_html(html):
    metrics = {k: None for k in ["revenue", "net_income", "assets", "liabilities", "equity"]}

    # Detectar si las cifras están en miles o millones
    scale = 1
    if re.search(r"in thousands", html, re.I):
        scale = 1_000
    elif re.search(r"in millions", html, re.I):
        scale = 1_000_000

    # Leer todas las tablas
    try:
        tables = pd.read_html(html)
    except:
        return metrics

    for df in tables:
        df = df.fillna("").astype(str)
        for idx, row in df.iterrows():
            row_text = " ".join(row.values[:2]).lower()
            for key, patterns in LABELS.items():
                if metrics[key] is None:
                    for pat in patterns:
                        if re.search(pat, row_text, re.I):
                            # Tomar el primer número grande de la fila
                            numbers = [parse_number(c) for c in row if parse_number(c) is not None]
                            if numbers:
                                metrics[key] = numbers[0] * scale
                            break
    return metrics

# ----- GET FILINGS -----
def get_all_filings(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    j = requests.get(sub_url, headers=HEADERS, timeout=30).json()

    def pack_from_recent(recent_dict):
        return pd.DataFrame({
            "accession": recent_dict["accessionNumber"],
            "form": recent_dict["form"],
            "filed": recent_dict["filingDate"],
            "primary": recent_dict["primaryDocument"]
        })

    recent_df = pack_from_recent(j["filings"]["recent"])
    older_parts = []
    for f in j.get("filings", {}).get("files", []):
        url = f"https://data.sec.gov/submissions/{f['name']}"
        y = requests.get(url, headers=HEADERS, timeout=30).json()
        if "filings" in y and "recent" in y["filings"]:
            older_parts.append(pack_from_recent(y["filings"]["recent"]))
        time.sleep(PAUSE)

    all_filings = pd.concat([recent_df] + older_parts, ignore_index=True)
    all_filings["filed"] = pd.to_datetime(all_filings["filed"]).dt.strftime("%Y-%m-%d")
    return all_filings.sort_values("filed")

# ----- MAIN -----
def main():
    cik_int = int(CIK)
    filings = get_all_filings(CIK)
    filings = filings[filings["form"].isin(FORMS)]
    filings = filings[filings["filed"].apply(in_range)]

    out_rows = []
    for _, r in filings.iterrows():
        url = pick_primary_doc_url(cik_int, r["accession"], r["primary"])
        try:
            html = requests.get(url, headers=HEADERS, timeout=30).text
            m = extract_metrics_from_html(html)
            out_rows.append({
                "filing_date": r["filed"],
                "form": r["form"],
                "revenue": m["revenue"],
                "net_income": m["net_income"],
                "total_assets": m["assets"],
                "total_liabilities": m["liabilities"],
                "shareholders_equity": m["equity"]
            })
            time.sleep(PAUSE)
        except:
            out_rows.append({
                "filing_date": r["filed"],
                "form": r["form"],
                "revenue": None,
                "net_income": None,
                "total_assets": None,
                "total_liabilities": None,
                "shareholders_equity": None
            })
            time.sleep(PAUSE)

    df = pd.DataFrame(out_rows).sort_values(["filing_date","form"])
    #df.to_csv("financials_data.csv", index=False)
    print(f"Saved {len(df)} filings to financials_data.csv")

if __name__ == "__main__":
    main()



/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_8761/3810232352.py:65: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_8761/3810232352.py:65: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_8761/3810232352.py:65: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_8761/3810232352.py:65: FutureWarning: Passing literal html to 'read_html' is deprecated and will b

Saved 44 filings to financials_data.csv
